<img src = "https://www.journalism.co.uk/assets/135/IiB_screenshot.jpg_resized_460_.jpeg" align = right width = 450>
<h1 align = left> Data 765: Python Fundamentals for Data Science</h1>
<h2 align = left> Lecture 13 | Exploratory Data Analysis (II) </h2>
<h3 align = left> Yinxian Zhang | QC Sociology</h3>

# Table of Contents

<div class = "alert alert-info">

1. [Data Preparation](#1)<br>
2. [Quick Review of Univariate Analysis](#2)<br>
3. [Bivariate Analysis](#3)<br>
    3.1 [Categorical v.s. categorical variables](#3.1)<br>
    3.2 [Categorical v.s. numerical variables](#3.2)<br>
    3.3 [Numerical v.s.  numerical variables](#3.3)<br>
4. [Saving Working Data](#4)<br>
    
</div>
<hr>

Let's continue to talk about EDA. We will import one new libary today:

In [ ]:
import pandas as pd                
import numpy as np   
import matplotlib.pyplot as plt                     
import seaborn as sns                               # a new viz library!
%matplotlib inline

---

# Data Preparation <a id=1></a>

We have explored the 2012 GSS dataset in the past couple of weeks. So let's quickly go over what we did and prepare the dataset for analysis and visualization.

In [ ]:
path = 'https://raw.githubusercontent.com/UC-MACSS/persp-analysis/master/assignments/exploratory-data-analysis/data/gss2012.csv'

df = pd.read_csv(path, header=0)           

We will skip the process of initial inspection and jump right into the variables of interest. At this point, you should be very familiar with this process.

In [ ]:
data =  df[['pres08',                              
            'polviews',                                  
            'id', 'age', 'sex', 'race', 'educ',                   
            'tvhours',
            'income06'               # a new variable for class demo
           ]]                        # do not overwrite! save cropped df to a new variable

Again, after reading in data, your primary task is to explore the dataset with the below questions in mind: 

- are there any **missing values**? Would the missing values influence my analysis?
- are there **illegal inputs** / special values that may affect my analysis? 
- are the variables in the desired **`dtype`s and formats**? 
- for numerical variables: what are the **measuring units**? are they discrete or continuous? 
- for categorial variables: how are they **coded**? are they **nominal or ordinal**?
- do I need to **recode** some of the variables?
- what are the **distributions** of the variables? Any **outliers or pecularities** that warrant further investigation?
- ...

We already explored each of the variables in the last couple of lectures, and we found a few issues:

1. clumsy variable names;
2. wrong `dtype` for `id`;
3. unordered categories under `polviews`;
3. too many (unordered) categories under `educ` and `income06`; and
4. a lot of missing values of `pres08`.

Now let's transform the data based on the findings from previous sessions. 

#### change variable names:

In [ ]:
data = data.rename(columns={'pres08':'pres', 
                            'polviews':'pol',
                            'income06': 'income'})    # for demo purposes only

data.columns

#### change dtypes:

In [ ]:
data['id'] = data.id.astype('object')

#### specify ordinal categorical variables:

In [ ]:
data.pol = pd.Categorical(data.pol, 
            categories=['ExtrmLib', 'Liberal', 'SlghtLib', 'Moderate', 'SlghtCons', 'Conserv', 'ExtrmCons'],   # with order
            ordered=True)

In [ ]:
data.pol.unique()              # double check

### ▲ The `pd.Categorical` command takes the original values of the categories as inputs. You CANNOT rename or merge categories here!  
### ▲ Please be very careful about typing. You cannot make any typos!

#### merge/collapse categories:

In [ ]:
data.educ.unique()                # 21 levels, un-ordered

In [ ]:
def recode(s):
    '''
    Input:
    s: string, the original values of the `educ` variable
    
    Output:
    recode: 4 categories representing different educational levels.
    '''
    if pd.isna(s):                  
        return np.nan                 # deal with missing values first
    
    if s == 'None':
        recode = 'no education'
    
    elif 'grade' in s:           
        recode = 'high school and below'    
        
    elif s in ['1 yr coll', '2 years', '3 years', '4 years']:      # manual typing is prone to mistakes
        recode = 'some college'
    
    elif s in ['5 years', '6 years', '7 years', '8 years']:
        recode = 'graduate level'
    
    else:
        print('error')                # helpful for debug: if there are other unexpected inputs, print it out
        print(s)
    
    return recode    

In [ ]:
# again, don't over write the orginal variable!

data['educ_recoded'] = data.educ.apply(recode)

In [ ]:
data[['educ', 'educ_recoded']]          # double check 

In [ ]:
# specify the ordering!!

data["educ_recoded"] = pd.Categorical(data.educ_recoded, 
                                      categories=["no education", "high school and below", "some college", "graduate level"], 
                                      ordered = True)


In [ ]:
data.educ_recoded.unique()         # double check

In [ ]:
# double check 

print(data.educ.isna().sum())
print(data.educ_recoded.isna().sum())

### Finally, a new approach of variable transformation: turning ordinal categorical variables with too many levels into numerical variables. But should use with caution!!

Some variables, such as `income`, are categorical **ordinal** variables with many levels (25 levels for income). So sometimes it makes sense to treat them as numerical variables in statistical analysis and visualization. 

However, to use them as numerical variables, we need to recode these variables:

In [ ]:
data.income.unique()                           

The variable features too many levels, and they are not organized / sorted at all. How to address this issue?

#### we can write a function and `.apply()`  to automatically sort the levels and convert them to numbers 

We can get rid of everything except for the digits, such as `'1000'`, `'2000'`, convert them to `int`, then calculate the average income of each level (or use the lower/higher bound as is).  

We can write a function to do the above, and `apply()` this function to a DataFrame or a Series, and it works element-wise on each row or column.

#### before writing the function, use a case to test your method first.

In [1]:
'$110000 TO $129999'.split('TO')

['$110000 ', ' $129999']

In [ ]:
test = '$110000 TO $129999'.split('TO')
print(test[0])
print(test[1])

In [ ]:
lower=''
for i in test[0]:
    if i.isdigit():
        lower+=i

higher=''
for j in test[1]:
    if j.isdigit():
        higher+=j
        
print(lower)
print(higher)

In [ ]:
(int(lower) + int(higher))/2

#### now, assemble the pieces and write the function:

In [ ]:
def income_convertion(s):          # apply on each row, so each s is a value of "income06"
    '''
    dosctrings omitted here
    BUT YOU SHOULD NOT OMIT THE DOCSTRINGS IN YOUR ASSIGNMENT
    '''
    
    if 'UNDER' in s:                # special case 1
        avg = 1000                      
    
    elif 'OVER' in s:               # special case 2
        avg = 150000
        
    elif 'TO' in s:                 # common cases                            
        income = s.split('TO')
        
        lower = ''   
        for i in income[0]:
            if i.isdigit():
                lower += i
                
        higher = ''
        for j in income[1]:
            if j.isdigit():
                higher += j
                
        avg = (int(lower) + int(higher))/2     # convert str to int, calculate avg
        
    else:
        print('error')                  # deal with unexpected errors
        print(s)
    
    return avg                          # return the avg incomebv  

In [ ]:
data['avg_inc'] = data.income.apply(income_convertion)

### Takeaway: `NaN` is of `float` type, and may cause problems in many operations. You may need to `dropna()` before you proceed.

In [ ]:
data['avg_income'] = data.income.dropna().apply(income_convertion)    

In [ ]:
# double check

data.income.value_counts()

In [ ]:
data.avg_income.nunique()

In [ ]:
check = sorted(data.avg_income.dropna().unique())
check

### ▲ Although being numerical, this variable contain only 25 unique values. They are discrete (rather than continuous), and do not contain many (unique) data points. This may create problems for statistical analysis and visualization, and many data scientists / sociologists would not recommend this type of conversion due to this issue. We will talk about it later. 

# Quick Review of Univariate Analysis <a id=2></a>

In our last lecture, we talked about how to do univariate analysis and obtain descriptive statistics & simple visualizations. The most important thing is to know the type of your variables:

### ▲ You have to use different analysis/visualization tools for each type of variable. 
But today we will NOT review the details of these different types of variables, and we will NOT go over each type of analysis and visualization tools again. Please refer to previous lecture notes on your own. **I would just like to remind you about a few common issues that I've seen**. 

<p style="color:red">Note, always remember to add appropriate titles and x/y axis labels to your graphs even if we may skip them here in the lecture notes! </p>

## Numerical variables

Histograms v.s. Boxplots: Both are for numerical variables **only**. \
Histograms show the full distribution of data points, while boxplots shows only important statistics: mix, max, 25%, 50% (median), 75% quartiles. 

### ▲ It's important to tweak the number of bins in `histogram`s: not too spiky, not too smoothed.

In [ ]:
# histograms:

data.age.plot(kind='hist', 
              bins=15)                                  # by default, 10 bins

## Categorical variables

Bar graphs are mostly for categorical variables, either nominal or ordinal. 

### ▲ bars need to be sorted when there are more than 2 categories.
### ▲ use descending/ascending order ONLY for *nominal* variables
### ▲ present the natural ranking of *ordinal* variables (i.e., use `pd.Categorical()` to specify the ranking beforehand).

In [ ]:
# nominal variable

data.groupby('race').size().sort_values(ascending=False).plot(kind='bar')   

In [ ]:
# ordinal variable

data.groupby('pol').size().plot(kind='barh')

# Bivariate Analysis <a id=3></a>

When it comes to the relationship between two variables, there are three possible scenarios:

1. categorical v.s. categorical
2. categorical v.s. numerical
3. numerical v.s. numerical

## Categorical v.s. Categorical <a id=3.1></a>
Let's talk about the first scenario first, using `pres` and `pol` as an example.

For bivariate analysis between two categorical variables, we typically rely on **crosstab analysis (aka contigency tables)**. We've used `groupby()` statistics to make contigency tables. We will also use `pivot_table()` because it provides a better view for analysis and visualization when we need to group by two or more variables. 

Both the `groupby()`/`pivot_table()` functions and the `.plot()` method would help you explore the bivariate relationships. 

#### Introducing `pivot_table()`

When your `groupby()` method involves multiple dimensions, the output may be confusing and hard to read. 

In [ ]:
# use groupby to get bivariate distribution:

data.groupby(['pol', 'pres']).size()

But when you use pivot table, it is organized in a clear and compact way! 

In [ ]:
# pivot table provides a better view and is better for visualization

data.pivot_table(columns='pol',                 # layout of the table 
                 index='pres', 
                 
                 values='id',                   # obtain headcount
                 aggfunc='count',
                 fill_value = 0
                 )  

# returns a dataframe, can be saved for future use!

The above table can be called a **"contingency table"** (please refer to your stats classes for a review). 

Therefore, after you create the contingency table with `groupby` or `pivot_table`, you can save the dataframe to a variable and pass it to certain functions from other packages for other statistical analysis, e.g. `scipy` for [chi-square tests](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html).

We will talk more about this in the next session.

For visualization, we often use **grouped bar graphs** to visualize the distribution over two categorical variables. 

In [ ]:
data.pivot_table(columns="pol",
                 index="pres",      
                 
                 values="id",
                 aggfunc='count',
                 fill_value=0).plot(kind='bar')

# Wrong graph. Ugly and not effective

The plot is quite...ugly, and ineffective in some sense (the color choices are really bad). 

**If this plot is not for your personal use, you need to make it aesthetically pleasing and effective for communication** -- these are more advanced topics of data visualization.  

### An important aspect of data visualization is **the layout** of your plots. 

You can draw very different plots based on exactly the same data. And **they are different not only in terms of the art, but also in terms of the specific messages that they deliver.** This influences what kind of stories you can tell!

For instance, the above grouped bar can be converted to a stacked bar graph. Can you compare the two plots and tell what different messages they deliver?

In [2]:
data.pivot_table(columns="pres",
                 index="pol",                      # swap cols and rows
                 
                 values="id",
                 aggfunc='count',
                 fill_value=0).plot(kind='bar',
                                    #stacked=True  # use stacked bars 
                                    )    

# WRONG graph. Unbalanced categories, hard to compare and could be misleading

NameError: name 'data' is not defined

Because of the unbalanced data distribution over political view categories (some categories have over 350 people while some have less than 50), the plot does not show clear & comparable patterns of voting decisions in different ideological groups. 

**To solve this problem, we can further tweak the plot and show percentages instead of counts.**

A easy way to do this is to use a function, `pd.crosstab`, whose syntax is very similiar to that of `df.pivot_table`.

In [ ]:
pd.crosstab(index=data.pol, 
            columns=data.pres, 
            
            values=data.pol, 
            aggfunc='count', 
            
            normalize='index'           # convert count to percentages, check docstring for other methods of normalization
           )                      

# returns a dataframe, can be saved for future use!

In [ ]:
pd.crosstab(index=data.pol, 
            columns=data.pres, 
            
            values=data.id, 
            aggfunc='count', 
            
            normalize='index').plot(kind='bar', 
                                    stacked=True)      # correct graph

In [ ]:
# edit the graph to make it effective and aesthetically pleasing.

fig = pd.crosstab(index=data.pol, 
                  columns=data.pres, 
            
                  values=data.id, 
                  aggfunc='count', 
            
                  normalize='index').plot(kind='barh', 
                                    stacked=True)

# change the location of legend
fig.legend(loc='lower right')


# always remember to add titles and labels once you finalze the graph!
plt.title('Distribution of 2008 Voting Decisions (MacCain v.s. Obama), by Political View, 2012 GSS (N=1187)')
plt.xlabel('percentage of voters')       
plt.ylabel('political views')

Check out more examples and code on [bar plots with pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.bar.html).


### ▲ grouped bars can be applied in most scenarios to visualize the distribution over two categorical variables.
### ▲ stacked bars are only good for binary variables (e.g. democrats v.s. republicans) where each bar has only two colored sections and add up to 100%.

## Categorical v.s. Numerical <a id=3.2></a>

Then, let's explore the relationship between one categorical and one numerical variables. Usually this means the distribution of a numerical variable by a categorical variable. Therefore, we can use **the same tools we just learned** to visualize the distributions of numerical variables, i.e., boxplot and histograms. 

Below is an example of the distribution of averge income (numerical variable) by `pres` (categorical variable), in numbers first:

In [ ]:
# groupby a categorical variable, then obtain stats for a numerical variable

data.groupby('pres')['avg_income'].agg(['mean', 'median', 'max', 'min', 'count'])

Now we can visualize these numbers in a more intuitive way. Note that the visualization methods are now **applied to the entire dataframe rather than just one column**, because we hope to involve multiple variables now. 

In [ ]:
# apply to dataframe instead of one col
data.boxplot(column='avg_income',                    # column: y axis (numerical) 
             by='pres')                              # by: x axis (categorical)         


plt.title('Distribution of household income by 2008 voting decisions, 2012 GSS (N=798)')
plt.xlabel('')
plt.ylabel('household income')
plt.suptitle('')

We can also do side-by-side histograms to visualize the same variables:

In [ ]:
data.hist(column='avg_income',
          by='pres', 
          sharex=True)                         # apply to dataframe instead of one col

## Numerical v.s. Numerical   <a id=3.3></a>

Lastly, to visualize the relationship between two numerical variables, we can use a scatterplot. 

But note that scatterplots using numerical **discrete** variables may look worse and less effective than those with **continuous** variables. 

Let's use `avg_income` and `tvhours` for a simple illustration. First, we can quickly check the correlation coefficient between the two variables:

In [ ]:
data.avg_income.corr(data.tvhours)                 # by default, return pearson's r

# Please check documentation for different methods used to compute correlation coefficients
# for significance levels, you can use other packages like "scipy.stats"

Then, we can make a scatterplot to show the relationship:

In [ ]:
data.plot.scatter(x='avg_income', y='tvhours')                   # both are discrete variables

plt.title('Distribution over household income and TV-watching Hours, 2012 GSS (N=1296)')
plt.xlabel('house hold income')
plt.ylabel('# of hours watching TV')

The plot does not clearly demonstrate the negative relationship between `avg_income` and `tvhours` as suggested by the `corr()` function. 

Moreover, due to the **discrete** nature of both variables, the plot suffers from a problem called **"overplotting"**. This issue may distort the data and make it hard for us to tell **how many observations there are, because the dots are overlapped right on top of each other.**

Luckily, we can adjust **transparency level** or **use jittering** to partly solve the problem with a new visualization library, `seaborn`(sns), which also fits a regression line to nicely demonstrate the relationship. But these techiniques introduce new problems too. 

In [ ]:
sns.lmplot(data=data, 
           x="avg_income", 
           y="tvhours", 
           
           #x_jitter=0.2,                     # adjust the degree of jittering 
           #y_jitter=0.2,
           #scatter_kws={'alpha':0.3},        # adjust the transparency level
           #height=8                          # adjust the size of the figure
          )

plt.title('Distribution over household income and TV-watching Hours, 2012 GSS (N=1296)')
plt.xlabel('household income')
plt.ylabel('# of hours watching TV')

### ▲ However, too much jittering will distort the data! 

<div class = "alert alert-warning">

Note: `seaborn` is a visualization library that supports easy and convenient plotting. You can make bar plots, histograms, boxplots and many more types of plots with `seaborn`, which are often more aesthetically pleasing than the default designs of `matplotlib`.  We will use `seaborn` to draw more plots in the next lecture.

To see more example of `seaborn`, please go to the [example gallary in its documentation](https://seaborn.pydata.org/examples/index.html). 
    
</div>

---
Copyright &copy; 2024 Yinxian Zhang | Department of Sociology | City University of New York, Queens College